<a href="https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Basil-Maqbool/flyrank-internship-assignment1/blob/main/work/notebooks/w05_model.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Basil-Maqbool/flyrank-internship-assignment1"
REPO_DIR = "flyrank-internship-assignment1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    target_dir = f"{REPO_DIR}/work/notebooks"
    if os.path.basename(os.getcwd()) != "notebooks":
        os.chdir(target_dir)
print("Ready! Current Working Directory:", os.getcwd())

Ready! Current Working Directory: /content/flyrank-internship-assignment1/work/notebooks


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice: Random Forest Classifier.
Our goal is to create a "which first?" ranking queue. A classifier's probability output gives us the exact scoring needed to rank content, which we can evaluate at Precision@50. A Random Forest is robust, requires minimal scaling, and allows us to easily extract feature importances to see what the model is leaning on, ensuring it stays honest

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design: Grouped Train/Test Split by client_id.
The dataset documentation explicitly warns that client_id should only be used for grouping and splits, never as a feature. If we do a standard random split, rows from the same client could end up in both the training and testing sets, causing the model to memorize client-specific behavior (data leakage). Grouping by client_id ensures the model is evaluated on entirely unseen clients.

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

# 1. Load Data
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# 2. Create target proxy (avoiding the label trap)
# The label is derived from trend_direction, so trend_pct and trend_direction can NEVER be features.
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 3. Define Features (Keeping it simple and honest)
features = ['content_age_days', 'days_since_last_update', 'impressions_90d', 'word_count']
df = df.dropna(subset=features + ['is_declining_label', 'client_id'])

X = df[features]
y = df['is_declining_label']
groups = df['client_id']

# 4. Grouped Split by client_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

# 5. Train the Random Forest Model (Fixing the random seed for reproducibility)
rf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

# 6. Generate Predictions (Using classifier probabilities for ranking)
test_df = X_test.copy()
test_df['actual_label'] = y_test
test_df['model_probability'] = rf.predict_proba(X_test)[:, 1]

# 7. Apply Week-4 Baseline Logic to the SAME split
def apply_baseline(row):
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500: return 100
    if row['days_since_last_update'] >= 90 and row['impressions_90d'] >= 100: return 50
    return 0

test_df['baseline_score'] = test_df.apply(apply_baseline, axis=1)

# 8. Evaluate Metric: Precision@50
top_50_model = test_df.sort_values('model_probability', ascending=False).head(50)
top_50_baseline = test_df.sort_values(['baseline_score', 'impressions_90d'], ascending=[False, False]).head(50)

model_p50 = top_50_model['actual_label'].mean()
baseline_p50 = top_50_baseline['actual_label'].mean()

print("--- THE COMPARISON TABLE ---")
print(f"Base Rate (Majority Class): {y.mean():.2f}")
print(f"Week 4 Baseline Precision@50: {baseline_p50:.2f}")
print(f"Random Forest Precision@50:   {model_p50:.2f}")

# 9. Extract Feature Importances for Error Analysis
importances = pd.DataFrame({'feature': features, 'importance': rf.feature_importances_})
print("\n--- FEATURE IMPORTANCES ---")
display(importances.sort_values('importance', ascending=False))

--- THE COMPARISON TABLE ---
Week 4 Baseline Precision@50: 0.32
Random Forest Precision@50:   0.48

--- FEATURE IMPORTANCES ---


,feature,importance
2,impressions_90d,0.659397
0,content_age_days,0.140702
3,word_count,0.133984
1,days_since_last_update,0.065917


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error Analysis & Interpretation:
The feature importance table shows that the model leans overwhelmingly on impressions_90d (65% importance), followed by content_age_days and word_count. Interestingly, days_since_last_update was the least important feature. Crucially, none of these features are suspiciously perfect, confirming we successfully avoided the trend_pct leakage trap.  Where the model is wrong:

Zero-Rank Pages: The model might penalize pages heavily for having zero traffic, but an avg_position = 0 actually means "no data," not rank zero.  

Static Intent: It predicts declines on older content based on content_age_days, but if a page is a simple glossary definition, it never needs an update regardless of age.

Missingness Injection: Some content types have naturally missing word counts. The model might misinterpret a missing word count as a signal of poor quality rather than just a specific formatting type (like an image gallery)

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.